# 第 24 节：多智能体架构与 Plan-and-Execute 规划落地

> 前置课程：第 22 节（对接大模型 + Function Calling + MCP 工具）、第 23 节（技能池）
> 对应项目：SmartBrief（`smartbrief/`）

---

## 课程导航

| 部分 | 内容 | 对应文件 |
|------|------|----------|
| **Part 1** | 课程导入：从一个需求说起 | — |
| **Part 2** | 环境准备与项目结构 | `requirements.txt` |
| **Part 3** | 单智能体的痛点：为什么需要多智能体 | — |
| **Part 4** | 共享状态 AgentState：多智能体的"白板" | `graph/state.py` |
| **Part 5** | Planner 演进：从硬编码计划到 LLM 规划 | `agents/planner.py` |
| **Part 6** | Scheduler 演进：从顺序执行到依赖派发 | `agents/scheduler.py` |
| **Part 7** | ReAct Researcher 演进：让 Agent 会"思考-行动-观察" | `agents/react_researcher.py` |
| **Part 8** | ReAct Writer 演进：从"空手写"到"带着资料写" | `agents/react_writer.py` |
| **Part 9** | LangGraph 工作流：把节点串成一张图 | `graph/workflow.py` |
| **Part 10** | SQLite 持久化：让任务"活下来" | `db/database.py` + `db/task_repository.py` |
| **Part 11** | API 与前端闭环 | `api/routes.py` + `frontend/src/views/` |
| **Part 12** | 端到端演示：完整跑一遍（模拟 LLM） | 多模块联合 |

---

> 💡 **面向学员**：本教程不是"背代码"，而是让你经历**项目开发构思的整个过程**——
> 每个核心设计都遵循：**原始问题 → 粗暴解法 → 遇到痛点 → 重构优化 → 最终代码**。
> 建议按顺序逐 Cell 运行，观察每一次的输出结果。

---
## Part 1：课程导入 —— 从一个需求说起
---

### 1.1 产品经理来了

你是 SmartBrief 项目的开发工程师。这天产品经理走过来，抛给你一个需求：

> "用户输入一个主题，比如『智能手表 2025 市场』，系统就要自动产出一份**市场调研报告**。
> 我们已经有 AI 对话、工具调用、技能池了，你把这件事做出来。"

你接到这个任务，第一反应是什么？

> 🤔 **暂停思考**：如果是你，你会怎么实现"输入关键词 → 自动产出报告"？

In [1]:
# ============================================================
# Cell 0：v0 —— 第一反应：让大模型一口气写完
# ============================================================

print("=" * 55)
print("  v0：最直觉的写法 —— 一个大 Prompt 直接生成报告")
print("=" * 55)
print()

print("  伪代码：")
print("  async def generate_report_v0(topic):")
print("      messages = [{'role': 'user',")
print("                    'content': f'请写一份关于 {topic} 的市场调研报告'}]")
print("      resp = await chat(messages)")
print("      return resp.choices[0].message.content")
print()

print("  [问题] 这份报告的质量如何？")
print("     1. 大模型没有联网搜索，数据和事实全靠'编'")
print("     2. 一次性生成超长内容，容易跑题、结构混乱")
print("     3. 过程完全不可见，出了问题无法排查")
print("     4. 所有逻辑揉在一个 Prompt 里，无法复用")

  v0：最直觉的写法 —— 一个大 Prompt 直接生成报告

  伪代码：
  async def generate_report_v0(topic):
      messages = [{'role': 'user',
                    'content': f'请写一份关于 {topic} 的市场调研报告'}]
      resp = await chat(messages)
      return resp.choices[0].message.content

  [问题] 这份报告的质量如何？
     1. 大模型没有联网搜索，数据和事实全靠'编'
     2. 一次性生成超长内容，容易跑题、结构混乱
     3. 过程完全不可见，出了问题无法排查
     4. 所有逻辑揉在一个 Prompt 里，无法复用


### 1.2 拆解：真实公司是怎么做报告的？

一家咨询公司写一份市场调研报告，从来不是一个人干的：

| 角色 | 职责 | 类比 |
|------|------|------|
| **项目经理** | 把大任务拆成小步骤（先查什么、再查什么、最后写什么） | Planner |
| **调度员** | 按依赖关系把步骤派给对应的人 | Scheduler |
| **研究员** | 联网搜索、收集数据，**边查边想** | ReAct Researcher |
| **写手** | 基于研究资料撰写报告章节 | ReAct Writer |

> 生活类比：开餐厅 🍜
> - 老板（Planner）定菜单：先买菜 → 再备菜 → 最后出菜
> - 后厨主管（Scheduler）派单：切菜工、炒菜工各司其职
> - 采购员（Researcher）去市场进货，发现缺什么再补买
> - 主厨（Writer）把食材做成菜

**多智能体协作** = 让多个各司其职的 Agent 通过**共享状态**协同完成一个大任务。

### 1.3 本节课的"开发旅程"

```
拆解线：一个大Prompt(质量差) -> 拆成多个Agent -> 共享State -> LangGraph编排
规划线：硬编码计划(死板)   -> LLM生成计划 -> 容错解析 -> 降级计划
执行线：搜索一次(不够)     -> ReAct循环   -> Thought/Action/Observation
持久化：内存(重启即丢)     -> SQLite落库  -> 任务列表/详情可查
```

---
## Part 2：环境准备与项目结构
---

### 2.1 检查 Python 与核心依赖

第 24 节引入 **LangGraph** 作为多智能体编排框架。**硬性要求**：

- `langchain >= 1.0`（**严禁使用 1.0 之前的版本**）
- `langchain-core >= 1.0`（LangGraph 1.x 依赖其内部模块）
- `langgraph >= 1.0`

LangGraph 1.x 需要 `langchain_core.language_models._compat_bridge`（仅 langchain-core 1.x 提供），
如果环境里是旧的 langchain 0.x + core 0.x，import LangGraph 会直接报 `ModuleNotFoundError`。

In [2]:
# ============================================================
# Cell 1：检查 Python 与 LangChain / LangGraph 版本
# ============================================================
import sys
import importlib

print("=" * 55)
print("  Python 与多智能体依赖检查")
print("=" * 55)
print(f"  Python 版本: {sys.version.split()[0]}")

required = ["langchain", "langchain_core", "langgraph", "openai", "dotenv"]
for pkg in required:
    try:
        mod = importlib.import_module(pkg)
        ver = getattr(mod, "__version__", "?")
        print(f"  [OK] {pkg:14s} {ver}")
    except ImportError:
        print(f"  [NO] {pkg} 未安装")

print()
try:
    import langchain
    major = int(langchain.__version__.split(".")[0])
    if major >= 1:
        print(f"  [通过] langchain {langchain.__version__} >= 1.0，符合要求")
    else:
        print(f"  [警告] langchain {langchain.__version__} 是 1.0 之前版本，严禁使用！")
        print("          执行: pip install -U langchain langchain-core langgraph")
except Exception as e:
    print(f"  [提示] 无法读取 langchain 版本: {e}")

  Python 与多智能体依赖检查
  Python 版本: 3.10.20
  [OK] langchain      1.3.16
  [OK] langchain_core 1.6.0
  [OK] langgraph      ?
  [OK] openai         2.46.0
  [OK] dotenv         ?

  [通过] langchain 1.3.16 >= 1.0，符合要求


### 2.2 项目结构一览（第 24 节新增部分）

```
smartbrief/backend/
├── agents/                    # 多智能体（第 24 节新增）
│   ├── planner.py             #   Planner：生成执行计划
│   ├── scheduler.py           #   Scheduler：按依赖派发步骤
│   ├── react_researcher.py    #   ReAct Researcher：Thought→Action(web_search)→Observation
│   └── react_writer.py        #   ReAct Writer：Thought→Action(llm_gen)→Observation
├── graph/                     # LangGraph 工作流（第 24 节新增）
│   ├── state.py               #   AgentState（多智能体共享状态）
│   └── workflow.py            #   build_graph() / run_task()
├── db/                        # SQLite 持久化（第 24 节新增）
│   ├── database.py            #   连接管理 + 建表
│   └── task_repository.py     #   任务/日志 CRUD
├── api/routes.py              # 新增 /generate、/tasks、/tasks/{id}/status
├── data.db                    # SQLite 业务库（tasks / task_logs）
└── llm/client.py              # chat()（第 22 节已有，多智能体复用）

frontend/src/views/
├── Home.vue                   # 任务提交页（新增）
├── Tasks.vue                  # 任务列表页（新增）
└── TaskDetail.vue             # 任务详情：Plan看板 + ReAct日志 + 报告草稿/导出（新增）
```

本节课重点走读 `agents/`、`graph/`、`db/` 三个目录。先让 notebook 能找到 backend 代码。

In [3]:
# ============================================================
# Cell 2：把 backend 加入 sys.path（供后续源码走读使用）
# ============================================================
import os
import sys

# 兼容两种运行位置：
#   1) notebook 在 proj03 根目录运行时 -> backend = ./smartbrief/backend
#   2) notebook 在 smartbrief 目录运行时 -> backend = ./backend
BACKEND = os.path.join(os.getcwd(), "smartbrief", "backend")
if not os.path.isdir(BACKEND):
    BACKEND = os.path.join(os.getcwd(), "backend")
if os.path.isdir(BACKEND):
    if BACKEND not in sys.path:
        sys.path.insert(0, BACKEND)
    print(f"  [OK] backend 已加入 sys.path:\n      {BACKEND}")
else:
    print(f"  [提示] 未找到 backend 目录（已尝试 ./smartbrief/backend 与 ./backend）")
    print(f"         请确认 notebook 与 smartbrief/backend 的相对位置正确")

# 验证能否导入 LangGraph
try:
    from langgraph.graph import StateGraph, START, END
    print("  [OK] 从 langgraph 导入 StateGraph / START / END 成功")
except ImportError as e:
    print(f"  [NO] LangGraph 导入失败: {e}")
    print("       请先: pip install -U langchain langchain-core langgraph")

  [OK] backend 已加入 sys.path:
      c:\Users\13691\Documents\Code\260601-GPKJ\05_multi-agent\proj04\smartbrief\backend
  [OK] 从 langgraph 导入 StateGraph / START / END 成功


---
## Part 3：单智能体的痛点 —— 为什么需要多智能体
---

### 3.1 一个 Agent 的局限

上一节课我们有了 `chat()`（单智能体对话）和 `web_search`（MCP 工具）。
但"生成一份报告"和"回答一个问题"是两回事：

| 需求 | 单智能体（一次调用） | 多智能体（分工协作） |
|------|---------------------|----------------------|
| 联网查真实数据 | ❌ 全靠模型记忆 | ✅ Researcher 调 web_search |
| 过程可控/可回溯 | ❌ 黑盒 | ✅ ReAct 日志逐步可见 |
| 超长任务 | ❌ 一次生成质量差 | ✅ 分步执行、逐步落库 |
| 扩展性 | ❌ 改一处全改 | ✅ 加一个 Agent 即可 |

### 3.2 关键设计一：共享状态 State

多个 Agent 怎么沟通？—— **共享一张"白板"**（一个 dict）。

```
┌────────────────────────────────────────────────────┐
│                 AgentState（白板）                    │
│  topic: 智能手表 2025 市场                          │
│  plan: [ {id:1, researcher, ...}, {id:2, ...} ]   │
│  research_data: { step_1: {...}, step_2: {...} }  │
│  draft_content: "## 市场规模\n2025年约300亿美元..." │
│  react_logs: [ {type: Thought, ...}, ... ]         │
└────────────────────────────────────────────────────┘
  Planner写 → Scheduler读+写 → Researcher写 → Writer读+写
```

每个节点**读白板、改白板**，改完把增量返回给框架，框架合并进共享状态。
这就是 LangGraph 的核心思想。

In [4]:
# ============================================================
# Cell 3：v1 —— 手写一张"白板"（先用 TypedDict 定义）
# ============================================================
from typing import TypedDict, List, Dict, Optional

class AgentState(TypedDict):
    """多智能体共享状态（白板）。

    字段分四组：
    - 任务信息：task_id / topic
    - Plan：plan / current_step_index / completed_steps
    - ReAct 日志：react_logs
    - 数据：research_data / draft_content / final_report / error
    """
    task_id: str
    topic: str
    plan: List[Dict]                # [{"id":1, "agent":"researcher", "task":"...", "depends_on":[]}]
    current_step_index: int
    completed_steps: List[int]
    react_logs: List[Dict]          # [{"type":"Thought/Action/Observation", "content":"...", "step_id":1}]
    research_data: Dict
    draft_content: str
    final_report: Optional[str]
    error: Optional[str]

# 初始化一张白板
state = AgentState(
    task_id="task_demo_01",
    topic="智能手表 2025 市场",
    plan=[],
    current_step_index=0,
    completed_steps=[],
    react_logs=[],
    research_data={},
    draft_content="",
    final_report=None,
    error=None,
)
print("  [OK] 白板已创建，当前字段：")
for k, v in state.items():
    print(f"     {k:20s} = {v}")

  [OK] 白板已创建，当前字段：
     task_id              = task_demo_01
     topic                = 智能手表 2025 市场
     plan                 = []
     current_step_index   = 0
     completed_steps      = []
     react_logs           = []
     research_data        = {}
     draft_content        = 
     final_report         = None
     error                = None


### 3.3 看看项目里的最终代码

我们手写的这张白板，就是项目里 `graph/state.py` 的真实定义。

In [5]:
# ============================================================
# Cell 4：走读 graph/state.py —— AgentState 最终代码
# ============================================================
import inspect

try:
    from graph.state import AgentState
    print("=" * 55)
    print("  graph/state.py 中的 AgentState")
    print("=" * 55)
    print(inspect.getsource(AgentState))
    print("  [提示] 和你 Cell 3 手写的几乎一样——这就是项目开发时的真实演进")
except Exception as e:
    print(f"  [提示] 导入失败（{e}）")
    print("         请确认已安装依赖，且 backend 目录可访问")

  graph/state.py 中的 AgentState
class AgentState(TypedDict):
    """多智能体共享状态（"白板"）。

    所有 Agent 节点都读写这张"白板"：
    - 节点函数接收完整 state，返回**增量 dict**（只包含要修改的字段）
    - LangGraph 框架负责把增量合并回共享状态

    字段分五组：
    1. 任务信息：task_id / topic
    2. Plan 进度：plan / current_step_index / completed_steps
    3. ReAct 日志：react_logs
    4. 产出数据：research_data / draft_content / final_report / error
    5. A2A 审核循环（第 25 节）：review_comments / review_history / iteration / max_iterations / passed / forced_pass
    """
    # ---------- 1. 任务信息 ----------
    task_id: str              # 任务唯一 id（由 API 层生成，形如 task_ab12cd34）
    topic: str                # 用户提交的研究主题（如 "智能手表 2025 市场"）

    # ---------- 2. Plan 进度 ----------
    plan: List[Dict]                # 步骤列表：[{"id":1, "agent":"researcher", "task":"...", "depends_on":[]}]
                                    # agent 取值：researcher（联网研究）/ writer（撰写章节）
    current_step_index: int         # 当前进度下标（= 已完成步骤数，供前端 Plan 看板定位）
    completed_steps: List[int]      # 已完成步骤 id 

---
## Part 4：Planner 演进 —— 从硬编码计划到 LLM 规划
---

### 4.1 v0：硬编码三步计划（能跑，但死板）

项目经理（Planner）第一次写计划，最简单的做法是**写死**。

In [6]:
# ============================================================
# Cell 5：v0 —— 硬编码计划
# ============================================================
def plan_v0(topic: str):
    """写死的三步计划：搜规模 -> 搜竞品 -> 写报告"""
    return [
        {"id": 1, "agent": "researcher", "task": f"搜索 {topic} 的市场规模", "depends_on": []},
        {"id": 2, "agent": "researcher", "task": f"搜索 {topic} 的主要竞品", "depends_on": [1]},
        {"id": 3, "agent": "writer", "task": f"撰写《{topic}》报告", "depends_on": [1, 2]},
    ]

plan = plan_v0("智能手表 2025 市场")
print("  写死的计划：")
for s in plan:
    print(f"     #{s['id']} [{s['agent']}] {s['task']}  依赖={s['depends_on']}")
print()
print("  [痛点] 换一个主题，比如'低空经济政策分析'：")
print("        也需要查政策、查产业链、查地方试点……但顺序完全不同！")
print("        硬编码无法适配任意主题，必须让 LLM 来'临场规划'。")

  写死的计划：
     #1 [researcher] 搜索 智能手表 2025 市场 的市场规模  依赖=[]
     #2 [researcher] 搜索 智能手表 2025 市场 的主要竞品  依赖=[1]
     #3 [writer] 撰写《智能手表 2025 市场》报告  依赖=[1, 2]

  [痛点] 换一个主题，比如'低空经济政策分析'：
        也需要查政策、查产业链、查地方试点……但顺序完全不同！
        硬编码无法适配任意主题，必须让 LLM 来'临场规划'。


### 4.2 v1：让 LLM 生成计划 —— 但 LLM 不靠谱

让 LLM 生成计划很简单，但大模型的输出**不保证是合法 JSON**：

- 可能用 Markdown 代码块包裹
- 可能夹杂解释文字
- 可能输出非法字段（agent 写错、task 为空）

所以我们要写一个**容错解析器** `parse_plan`。先看看它会遇到哪些"刁钻输出"。

In [7]:
# ============================================================
# Cell 6：v1 —— LLM 的"刁钻输出"演示
# ============================================================
import json

# 场景 A：LLM 老老实实返回 JSON（理想情况）
good = '[{"id": 1, "agent": "researcher", "task": "搜索市场规模", "depends_on": []}]'

# 场景 B：LLM 用 Markdown 代码块包裹（常见！）
with_fence = '```json\n[{"id": 1, "agent": "researcher", "task": "搜索市场规模", "depends_on": []}]\n```'

# 场景 C：LLM 在 JSON 前后加了废话
with_talk = '好的，以下是计划：\n[{"id": 1, "agent": "researcher", "task": "搜索市场规模", "depends_on": []}]\n希望对你有帮助'

for name, text in [("场景A 理想", good), ("场景B 代码块", with_fence), ("场景C 夹带废话", with_talk)]:
    try:
        data = json.loads(text)
        print(f"  [OK] {name}: json.loads 直接成功")
    except json.JSONDecodeError as e:
        print(f"  [NO] {name}: json.loads 直接失败 -> {e}")

print()
print("  [结论] 不能直接 json.loads，需要一个容错解析函数！")

  [OK] 场景A 理想: json.loads 直接成功
  [NO] 场景B 代码块: json.loads 直接失败 -> Expecting value: line 1 column 1 (char 0)
  [NO] 场景C 夹带废话: json.loads 直接失败 -> Expecting value: line 1 column 1 (char 0)

  [结论] 不能直接 json.loads，需要一个容错解析函数！


### 4.3 v2：自己写一个容错解析器

解析策略：
1. 去掉可能的 ```json ... ``` 代码块
2. 截取第一个 `[` 到最后一个 `]` 之间的内容再解析
3. 过滤非法步骤（agent 不在白名单、task 为空）

> 这就是项目里 `agents/planner.py` 中 `parse_plan()` 的设计思路。

In [8]:
# ============================================================
# Cell 7：v2 —— 手写容错解析器（对应项目 parse_plan）
# ============================================================
import json
import re
from typing import List, Dict

MAX_STEPS = 5

def parse_plan_v2(content: str) -> List[Dict]:
    """容错解析 LLM 输出的 Plan 步骤列表"""
    text = (content or "").strip()
    # 1. 去掉 Markdown 代码块包裹
    fence = re.search(r"```(?:json)?\s*(.*?)\s*```", text, re.DOTALL)
    if fence:
        text = fence.group(1).strip()
    # 2. 截取 [ ] 区间
    start, end = text.find("["), text.rfind("]")
    if start == -1 or end == -1:
        return []
    try:
        data = json.loads(text[start:end + 1])
    except json.JSONDecodeError:
        return []
    if not isinstance(data, list):
        return []
    # 3. 过滤非法步骤
    steps = []
    for item in data:
        if not isinstance(item, dict):
            continue
        step = {
            "id": int(item.get("id", len(steps) + 1)),
            "agent": item.get("agent", "researcher"),
            "task": str(item.get("task", "")).strip(),
            "depends_on": item.get("depends_on", []) or [],
        }
        if step["agent"] not in ("researcher", "writer") or not step["task"]:
            continue
        steps.append(step)
    return steps[:MAX_STEPS]

# 用三种"刁钻输出"测试
test_cases = {
    "理想 JSON":   '[{"id": 1, "agent": "researcher", "task": "搜索市场规模", "depends_on": []}]',
    "代码块包裹":  '```json\n[{"id": 1, "agent": "researcher", "task": "搜索市场规模", "depends_on": []}]\n```',
    "夹带废话":    '好的：\n[{"id": 1, "agent": "researcher", "task": "搜索市场规模", "depends_on": []}]\n请查收',
    "非法步骤":    '[{"id": 1, "agent": "黑客", "task": "乱来", "depends_on": []}, {"id": 2, "agent": "writer", "task": "撰写报告", "depends_on": []}]',
}
for name, content in test_cases.items():
    result = parse_plan_v2(content)
    print(f"  [{name}] 解析出 {len(result)} 步: {result}")

  [理想 JSON] 解析出 1 步: [{'id': 1, 'agent': 'researcher', 'task': '搜索市场规模', 'depends_on': []}]
  [代码块包裹] 解析出 1 步: [{'id': 1, 'agent': 'researcher', 'task': '搜索市场规模', 'depends_on': []}]
  [夹带废话] 解析出 1 步: [{'id': 1, 'agent': 'researcher', 'task': '搜索市场规模', 'depends_on': []}]
  [非法步骤] 解析出 1 步: [{'id': 2, 'agent': 'writer', 'task': '撰写报告', 'depends_on': []}]


### 4.4 看看项目里的最终代码

`parse_plan_v2` 就是 `agents/planner.py` 的真实逻辑。另外还有两个关键设计：

- **`fallback_plan(topic)`**：LLM 挂了/解析失败时的**降级计划**（保证流程永远能跑）
- **`planner_node(state)`**：LangGraph 节点函数，输入 state，返回 state 增量

> 🤔 **暂停思考**：如果 LLM 服务不可用，Planner 直接崩溃，整个任务就死了。
> 降级计划就是为了应对这种"外部依赖不可用"的容灾设计。

In [9]:
# ============================================================
# Cell 8：走读 agents/planner.py —— Planner 最终代码
# ============================================================
import inspect

try:
    from agents import planner
    print("=" * 55)
    print("  parse_plan 源码")
    print("=" * 55)
    print(inspect.getsource(planner.parse_plan))
except Exception as e:
    print(f"  [提示] 导入失败（{e}）")

  parse_plan 源码
def parse_plan(content: str) -> List[Dict]:
    """从 LLM 输出中解析 Plan 步骤列表。

    为什么需要"容错"？因为 LLM 的输出不可靠，常见三种情况：
    1. 用 ```json ... ``` 代码块包裹（最常见）
    2. 在 JSON 前后夹带解释文字（如"好的，以下是计划：..."）
    3. 步骤字段非法（agent 写错、task 为空）

    解析策略（由易到难逐级容错）：
    第一步：剥掉 Markdown 代码块包裹
    第二步：仍解析失败则截取第一个 [ 到最后一个 ] 再尝试
    第三步：过滤非法步骤（agent 不在白名单 / task 为空）
    最终：超出 MAX_STEPS 的步骤截断，防止 LLM 一次给太多
    """
    text = (content or "").strip()
    # 1) 去掉可能的 ```json ... ``` 包裹（正则 DOTALL 让 . 也能匹配换行）
    fence = re.search(r"```(?:json)?\s*(.*?)\s*```", text, re.DOTALL)
    if fence:
        text = fence.group(1).strip()
    try:
        data = json.loads(text)          # 场景A：直接就是合法 JSON
    except json.JSONDecodeError:
        # 场景B/C：截取 [ 到 ] 的区间（剥掉前后废话）
        start, end = text.find("["), text.rfind("]")
        if start == -1 or end == -1:
            return []                    # 连 [ ] 都没有，说明 LLM 完全跑题
        try:
            data = json.loads(text[start : end + 1])
        except json.JSONDec

In [10]:
# ============================================================
# Cell 9：继续走读 —— fallback_plan 与 planner_node
# ============================================================
try:
    from agents import planner
    print("=" * 55)
    print("  fallback_plan 源码（降级计划）")
    print("=" * 55)
    print(inspect.getsource(planner.fallback_plan))
    print()
    print("=" * 55)
    print("  planner_node 源码（LangGraph 节点）")
    print("=" * 55)
    print(inspect.getsource(planner.planner_node))
except Exception as e:
    print(f"  [提示] 导入失败（{e}）")

  fallback_plan 源码（降级计划）
def fallback_plan(topic: str) -> List[Dict]:
    """降级计划：LLM 挂了/解析失败时的"兜底方案"。

    教学要点：外部依赖（LLM）不可用时，整个流程不能死掉。
    这里给出一个固定三步计划（市场规模 → 竞品 → 撰写），
    保证任务永远能往下跑 —— 这是工程上"降级设计"的典型例子。

    依赖关系：
      step1（搜规模）无依赖
      step2（搜竞品）依赖 step1
      step3（写报告）依赖 step1、step2  —— 数据齐了才能写
    """
    return [
        {"id": 1, "agent": "researcher", "task": f"搜索 {topic} 的市场规模与整体概况", "depends_on": []},
        {"id": 2, "agent": "researcher", "task": f"搜索 {topic} 的主要竞品与头部玩家", "depends_on": [1]},
        {"id": 3, "agent": "writer", "task": f"撰写《{topic}》市场调研报告草稿", "depends_on": [1, 2]},
    ]


  planner_node 源码（LangGraph 节点）
async def planner_node(state: dict) -> dict:
    """LangGraph 节点：输入 topic → 输出 plan。

    节点约定：入参是完整 state，返回值是**增量 dict**（只含要更新的字段），
    LangGraph 会自动把增量合并回共享状态。

    执行步骤：
    1. 从 state 取研究主题 topic
    2. 用 SYSTEM_PROMPT + topic 组装消息，调用 LLM 生成计划文本
    3. LLM 调用失败 → 记录日志 + 走降级计划
    4. 解析 LLM 输出；解析不出有效计划 → 走降级计划
    5. 在日志最前面插入一条 Thought（"已生成执行计划"）

---
## Part 5：Scheduler 演进 —— 从顺序执行到依赖派发
---

### 5.1 v0：顺序傻跑的问题

计划有了，谁来执行？最直觉的写法是**按顺序一个个跑**。

In [11]:
# ============================================================
# Cell 10：v0 —— 顺序傻跑
# ============================================================
plan = [
    {"id": 1, "agent": "researcher", "task": "搜索市场规模", "depends_on": []},
    {"id": 2, "agent": "writer",     "task": "撰写报告",     "depends_on": [1]},
]

print("  v0 顺序傻跑：")
for step in plan:
    print(f"    执行 #{step['id']} [{step['agent']}] {step['task']}")
print()
print("  [问题] 如果计划是这样的：")
plan_bad = [
    {"id": 1, "agent": "writer",     "task": "撰写报告", "depends_on": [2]},
    {"id": 2, "agent": "researcher", "task": "搜索市场规模", "depends_on": []},
]
for step in plan_bad:
    print(f"    #{step['id']} [{step['agent']}] {step['task']}  依赖={step['depends_on']}")
print()
print("  [灾难] Writer 排在 Researcher 前面，写手拿不到数据！")
print("  [结论] 必须按 depends_on 依赖关系派发，而不是机械地按顺序跑。")

  v0 顺序傻跑：
    执行 #1 [researcher] 搜索市场规模
    执行 #2 [writer] 撰写报告

  [问题] 如果计划是这样的：
    #1 [writer] 撰写报告  依赖=[2]
    #2 [researcher] 搜索市场规模  依赖=[]

  [灾难] Writer 排在 Researcher 前面，写手拿不到数据！
  [结论] 必须按 depends_on 依赖关系派发，而不是机械地按顺序跑。


### 5.2 v1：依赖就绪判定

Scheduler 的核心是一个纯函数：**找出所有"依赖已满足且未完成"的步骤**。

In [12]:
# ============================================================
# Cell 11：v1 —— 依赖就绪判定（对应项目 _ready_steps）
# ============================================================
from typing import List, Dict

def _ready_steps_v1(plan: List[Dict], completed: List[int]) -> List[Dict]:
    """返回所有依赖已满足、尚未完成的步骤（按 id 升序）"""
    completed_set = set(completed)
    ready = []
    for step in plan:
        if step["id"] in completed_set:
            continue
        if all(dep in completed_set for dep in step.get("depends_on", [])):
            ready.append(step)
    return sorted(ready, key=lambda s: s["id"])

plan = [
    {"id": 1, "agent": "researcher", "task": "搜市场规模", "depends_on": []},
    {"id": 2, "agent": "researcher", "task": "搜竞品",     "depends_on": [1]},
    {"id": 3, "agent": "writer",     "task": "写报告",     "depends_on": [1, 2]},
]

for done in [[], [1], [1, 2], [1, 2, 3]]:
    ready = _ready_steps_v1(plan, done)
    print(f"  已完成={done} -> 就绪步骤={[s['id'] for s in ready]}")
print()
print("  [观察] 依赖没满足的步骤永远不会提前执行")
print("        全部完成时就绪列表为空 -> 调度结束")

  已完成=[] -> 就绪步骤=[1]
  已完成=[1] -> 就绪步骤=[2]
  已完成=[1, 2] -> 就绪步骤=[3]
  已完成=[1, 2, 3] -> 就绪步骤=[]

  [观察] 依赖没满足的步骤永远不会提前执行
        全部完成时就绪列表为空 -> 调度结束


### 5.3 看看项目里的最终代码

`_ready_steps_v1` 就是 `agents/scheduler.py` 的真实逻辑。
项目里 `scheduler_node()` 会**反复**调用就绪判定：每执行完一批步骤，重新计算就绪列表，
直到没有可执行步骤为止（相当于一个"while 有活干就派活"的循环）。

In [13]:
# ============================================================
# Cell 12：走读 agents/scheduler.py —— Scheduler 最终代码
# ============================================================
import inspect

try:
    from agents import scheduler
    print("=" * 55)
    print("  _ready_steps 源码")
    print("=" * 55)
    print(inspect.getsource(scheduler._ready_steps))
    print()
    print("=" * 55)
    print("  scheduler_node 源码（LangGraph 节点）")
    print("=" * 55)
    print(inspect.getsource(scheduler.scheduler_node))
except Exception as e:
    print(f"  [提示] 导入失败（{e}）")

  _ready_steps 源码
def _ready_steps(plan: List[Dict], completed: List[int]) -> List[Dict]:
    """返回所有"依赖已满足、尚未完成"的步骤（按 id 升序，保证确定性）。

    就绪判定规则：
    - 步骤的 id 不在 completed 中（还没执行过）
    - 步骤 depends_on 里的每个依赖 id 都在 completed 中（依赖全完成）

    为什么按 id 升序？保证多次调度结果一致（确定性），
    避免 LLM 计划里步骤乱序导致执行顺序不稳定。
    """
    completed_set = set(completed)               # 转 set 加速"依赖是否已完成"判断
    ready = []
    for step in plan:
        if step["id"] in completed_set:
            continue                             # 已完成的步骤跳过
        # all(...)：depends_on 为空列表时返回 True（无依赖 → 立即就绪）
        if all(dep in completed_set for dep in step.get("depends_on", [])):
            ready.append(step)
    return sorted(ready, key=lambda s: s["id"])  # 按 id 升序，保证确定性


  scheduler_node 源码（LangGraph 节点）
async def scheduler_node(state: dict) -> dict:
    """LangGraph 节点：调度执行所有可执行步骤。

    核心思想：**反复"取就绪步骤 → 派发执行 → 更新完成状态"**，
    直到没有可执行步骤为止（类似 while 有活干就派活）。

    派发规则：
    - researcher 步骤 → 调 react_researcher_run（ReAct 搜索循环）
   

---
## Part 6：ReAct Researcher 演进 —— 让 Agent 会"思考-行动-观察"
---

### 6.1 v0：搜索一次就交差

研究员的初版实现很简单：拿到子任务，搜一次，把结果丢回去。

In [14]:
# ============================================================
# Cell 13：v0 —— 搜索一次就交差
# ============================================================
async def research_v0(task: str):
    """初版：搜索一次，返回结果"""
    result = await call_tool("web_search", {"query": task})
    return result

print("  v0 伪代码：")
print("    搜索一次 -> 返回结果 -> 结束")
print()
print("  [问题1] 如果第一次搜索词不好（太宽泛），结果全是废话，没有补救")
print("  [问题2] 如果信息不够（比如只要 2025 年数据，结果却是 2022 年的），没有兜底")
print("  [问题3] 不知道什么时候该停（信息足够却还在搜 = 浪费）")
print()
print("  [结论] 研究员需要'边查边想'的能力 —— 这就是 ReAct 范式")

  v0 伪代码：
    搜索一次 -> 返回结果 -> 结束

  [问题1] 如果第一次搜索词不好（太宽泛），结果全是废话，没有补救
  [问题2] 如果信息不够（比如只要 2025 年数据，结果却是 2022 年的），没有兜底
  [问题3] 不知道什么时候该停（信息足够却还在搜 = 浪费）

  [结论] 研究员需要'边查边想'的能力 —— 这就是 ReAct 范式


### 6.2 ReAct 范式：Thought → Action → Observation

**ReAct（Reasoning + Acting）** 让 Agent 循环执行三个步骤：

```
Thought（思考）      -> "市场规模数据还不够新，我需要搜 2025 年的数据"
Action（行动）       -> {"action": "web_search", "query": "智能手表 2025 市场规模"}
Observation（观察）  -> 搜索引擎返回的结果
      │
      ▼ （把 Observation 喂回给 LLM，继续思考）
Thought（思考）      -> "2025 年市场规模 300 亿美元，够了，可以结束"
Action（行动）       -> {"action": "done"}
```

> 生活类比：菜鸟侦探 🕵️ 一次问完就写报告；老侦探会"观察到线索 → 调整方向 → 再查 → 直到真相大白"。

关键设计（对应 `react_researcher.py`）：

- `MAX_REACT_STEPS = 3`：最多行动 3 次，防止无限循环
- LLM 每轮输出一个 JSON 决策：`{"action": "web_search", "query": "..."}` 或 `{"action": "done"}`
- 每轮把 Thought / Action / Observation 追加到 `react_logs`（前端日志流的来源）
- LLM 不可用时**降级为直接搜索一次**（鲁棒性兜底）

In [15]:
# ============================================================
# Cell 14：v1 —— 用"假工具"演示 ReAct 循环（不联网、不调 LLM）
# ============================================================
import json

# ---- 假工具：返回预设结果 ----
FAKE_KB = {
    "智能手表 2025 市场规模": "2025 年全球智能手表市场规模约 300 亿美元，年增 15%",
    "智能手表 头部厂商": "Apple、华为、三星、小米占据前四",
}

def fake_search(query):
    for k, v in FAKE_KB.items():
        if k in query:
            return v
    return "未找到相关数据"

# ---- 假 LLM：模拟"思考"后的决策（真实项目里由 LLM 输出）----
def fake_llm_decide(history):
    # 简单规则：第一轮搜市场规模，第二轮搜竞品，第三轮结束
    if history == []:
        return {"action": "web_search", "query": "智能手表 2025 市场规模"}
    if history == ["searched_1"]:
        return {"action": "web_search", "query": "智能手表 头部厂商"}
    return {"action": "done"}

# ---- ReAct 循环（真实项目 react_researcher_run 的精简版）----
MAX_STEPS = 3

def react_loop_v1(task, max_steps=MAX_STEPS):
    logs = []
    history = []
    observations = []
    for round_no in range(1, max_steps + 1):
        decision = fake_llm_decide(history)
        if decision["action"] == "done":
            logs.append(("Thought", "信息已足够，结束本轮研究"))
            break
        query = decision["query"]
        logs.append(("Thought", f"第 {round_no} 轮：我需要搜索「{query}」"))
        logs.append(("Action", f"web_search(query={query!r})"))
        obs = fake_search(query)
        logs.append(("Observation", obs[:40] + "..."))
        observations.append({"query": query, "result": obs})
        history.append("searched_1")  # 简化：记录已搜索
    return logs, observations

# ---- 运行演示 ----
logs, observations = react_loop_v1("智能手表 2025 市场规模")
print("  ReAct 循环日志（模拟）：")
for i, (t, c) in enumerate(logs, 1):
    print(f"    {i}. [{t}] {c}")
print()
print("  收集到的观察数据：")
for obs in observations:
    print(f"    - {obs['result']}")
print()
print("  [收获] 每一次 Action 的结果都喂回给 Agent 继续思考")
print("         信息够了就 done，不会无限搜索")

  ReAct 循环日志（模拟）：
    1. [Thought] 第 1 轮：我需要搜索「智能手表 2025 市场规模」
    2. [Action] web_search(query='智能手表 2025 市场规模')
    3. [Observation] 2025 年全球智能手表市场规模约 300 亿美元，年增 15%...
    4. [Thought] 第 2 轮：我需要搜索「智能手表 头部厂商」
    5. [Action] web_search(query='智能手表 头部厂商')
    6. [Observation] Apple、华为、三星、小米占据前四...
    7. [Thought] 信息已足够，结束本轮研究

  收集到的观察数据：
    - 2025 年全球智能手表市场规模约 300 亿美元，年增 15%
    - Apple、华为、三星、小米占据前四

  [收获] 每一次 Action 的结果都喂回给 Agent 继续思考
         信息够了就 done，不会无限搜索


### 6.3 看看项目里的最终代码

`react_researcher.py` 把上面的循环变成了真实代码，关键点：

- **`parse_decision(content, fallback_query)`**：解析 LLM 的行动决策 JSON（同样容错处理代码块）
- **`react_researcher_run(task, step_id, logs)`**：真正的 ReAct 循环，最多 3 轮
- 每轮日志（Thought / Action / Observation）都带 `step_id`，方便前端按步骤归类展示

In [16]:
# ============================================================
# Cell 15：走读 agents/react_researcher.py —— ReAct Researcher
# ============================================================
import inspect

try:
    from agents import react_researcher
    print("=" * 55)
    print("  parse_decision 源码")
    print("=" * 55)
    print(inspect.getsource(react_researcher.parse_decision))
except Exception as e:
    print(f"  [提示] 导入失败（{e}）")

  parse_decision 源码
def parse_decision(content: str, fallback_query: str) -> dict:
    """解析 LLM 的行动决策 JSON（容错处理代码块 / 多余文字）。

    决策格式：{"action": "web_search", "query": "..."} 或 {"action": "done"}
    与 parse_plan 同理：LLM 输出不可靠，需要逐级容错。
    解析失败时的兜底：默认返回一次 web_search（用 fallback_query）。

    :param content: LLM 输出文本
    :param fallback_query: 解析失败时使用的默认搜索词（通常是子任务原文）
    """
    text = (content or "").strip()
    # 1) 剥掉 ```json ... ``` 代码块
    fence = re.search(r"```(?:json)?\s*(.*?)\s*```", text, re.DOTALL)
    if fence:
        text = fence.group(1).strip()
    try:
        data = json.loads(text)                 # 直接解析
    except json.JSONDecodeError:
        # 截取 { 到 } 的区间（剥掉前后废话）再试
        start, end = text.find("{"), text.rfind("}")
        if start == -1 or end == -1:
            return {"action": "web_search", "query": fallback_query}
        try:
            data = json.loads(text[start : end + 1])
        except json.JSONDecodeError:
            return {"action": "web_search", "

In [17]:
# ============================================================
# Cell 16：继续走读 —— react_researcher_run 主循环
# ============================================================
try:
    from agents import react_researcher
    print("=" * 55)
    print("  react_researcher_run 源码")
    print("=" * 55)
    print(inspect.getsource(react_researcher.react_researcher_run))
    print()
    print(f"  [提示] MAX_REACT_STEPS = {react_researcher.MAX_REACT_STEPS}")
except Exception as e:
    print(f"  [提示] 导入失败（{e}）")

  react_researcher_run 源码
async def react_researcher_run(task: str, step_id: int, logs: list) -> dict:
    """对单个 research 步骤执行 ReAct 循环（最多 MAX_REACT_STEPS 轮）。

    循环结构（每轮三拍）：
        拍1 Thought（思考）：LLM 输出决策文本
        拍2 Action（行动）：决策是 web_search → 执行 _run_search
        拍3 Observation（观察）：把搜索结果追加到 messages，供下一轮思考

    结束条件（满足其一）：
        1. LLM 决策为 done（信息已足够）
        2. 用完 MAX_REACT_STEPS 轮（防止无限循环）
        3. 搜索调用失败（带 error 直接 break）

    :param task: 该步骤的子任务描述
    :param step_id: 对应 plan 步骤 id（日志归属）
    :param logs: 共享 react_logs 列表（原地追加）
    :return: {"observations": [web_search 结果列表]}
    """
    observations = []                                   # 收集本轮所有搜索结果
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"研究子任务：{task}"},
    ]
    for round_no in range(1, MAX_REACT_STEPS + 1):
        # ---------- 拍1 Thought：让 LLM 思考并输出决策 ----------
        try:
            resp = await chat(messages)
            decision_text = resp.

---
## Part 7：ReAct Writer 演进 —— 从"空手写"到"带着资料写"
---

### 7.1 v0：让 LLM 空手写章节

写手初版：让 LLM 直接写，什么都不给。

In [18]:
# ============================================================
# Cell 17：v0 —— 空手写 vs 带资料写
# ============================================================
print("  v0 空手写：")
print("     chat('请写一份关于智能手表的报告章节')")
print()
print("  [问题] LLM 没有数据，只能靠训练记忆'编'，数字不可信")
print()
print("  v1 带资料写：")
print("     1. 先从 research_data 汇总出摘要文本")
print("     2. 把摘要塞进 Prompt：'请基于以下研究数据撰写……'")
print("     3. LLM 引用真实数字输出章节")
print()
print("  [关键] 研究数据可能很杂（多轮搜索的原始 JSON），")
print("         直接全量塞给 Writer 会爆上下文、还会让 LLM 抓不住重点。")
print("         所以要先做一步'压缩摘要'。")

  v0 空手写：
     chat('请写一份关于智能手表的报告章节')

  [问题] LLM 没有数据，只能靠训练记忆'编'，数字不可信

  v1 带资料写：
     1. 先从 research_data 汇总出摘要文本
     2. 把摘要塞进 Prompt：'请基于以下研究数据撰写……'
     3. LLM 引用真实数字输出章节

  [关键] 研究数据可能很杂（多轮搜索的原始 JSON），
         直接全量塞给 Writer 会爆上下文、还会让 LLM 抓不住重点。
         所以要先做一步'压缩摘要'。


### 7.2 v1：先压缩研究数据再撰写

项目里 `_summarize_research()` 负责把 `research_data` 压缩成可供 Writer 引用的 Markdown 摘要。

In [19]:
# ============================================================
# Cell 18：演示研究数据压缩（对应 _summarize_research 逻辑）
# ============================================================
import json

research_data = {
    "step_1": {"observations": [
        {"query": "智能手表 2025 市场规模",
         "result": {"content": "2025 年全球智能手表市场规模约 300 亿美元，年增 15%"}}
    ]},
    "step_2": {"observations": [
        {"query": "智能手表 头部厂商",
         "result": {"content": "Apple、华为、三星、小米占据前四"}}
    ]},
}

def summarize_v1(research_data: dict) -> str:
    parts = []
    for key, value in research_data.items():
        parts.append(f"### {key}")
        if isinstance(value, dict) and "observations" in value:
            for obs in value["observations"]:
                text = json.dumps(obs.get("result", ""), ensure_ascii=False)[:300]
                parts.append(f"- 关键词 {obs.get('query', '')}：{text}")
        else:
            parts.append(f"- {json.dumps(value, ensure_ascii=False)[:300]}")
    return "\n".join(parts)

print("  压缩后的研究摘要（喂给 Writer 的 Prompt 素材）：")
print("  " + "-" * 50)
print(summarize_v1(research_data))
print("  " + "-" * 50)
print()
print("  [设计] 每个步骤的数据用一个 ### 小节组织")
print("         关键字段限制 300 字，防止上下文爆炸")

  压缩后的研究摘要（喂给 Writer 的 Prompt 素材）：
  --------------------------------------------------
### step_1
- 关键词 智能手表 2025 市场规模：{"content": "2025 年全球智能手表市场规模约 300 亿美元，年增 15%"}
### step_2
- 关键词 智能手表 头部厂商：{"content": "Apple、华为、三星、小米占据前四"}
  --------------------------------------------------

  [设计] 每个步骤的数据用一个 ### 小节组织
         关键字段限制 300 字，防止上下文爆炸


In [20]:
# ============================================================
# Cell 19：走读 agents/react_writer.py —— ReAct Writer
# ============================================================
import inspect

try:
    from agents import react_writer
    print("=" * 55)
    print("  _summarize_research 源码")
    print("=" * 55)
    print(inspect.getsource(react_writer._summarize_research))
    print()
    print("=" * 55)
    print("  react_writer_run 源码（行动 = llm_gen）")
    print("=" * 55)
    print(inspect.getsource(react_writer.react_writer_run))
except Exception as e:
    print(f"  [提示] 导入失败（{e}）")

  _summarize_research 源码
def _summarize_research(research_data: dict) -> str:
    """把研究数据压缩成供 Writer 引用的 Markdown 摘要。

    为什么要压缩？
    1. 研究数据是原始 JSON（多轮搜索的完整返回），直接塞给 LLM 会爆上下文
    2. 摘要按步骤分组（### step_1 / step_2 ...），让 LLM 清楚每条数据来自哪一步
    3. 每个关键字段限 300 字，防止上下文爆炸

    :param research_data: {"step_1": {"observations": [...]}, ...}
    :return: 压缩后的 Markdown 摘要文本
    """
    if not research_data:
        return "（暂无研究数据）"
    parts = []
    for key, value in research_data.items():         # 遍历每个步骤的研究结果
        parts.append(f"### {key}")                   # 用标题区分步骤
        if isinstance(value, dict) and "observations" in value:
            for obs in value["observations"]:        # 展开该步骤下的每条搜索结果
                parts.append(f"- 关键词 {obs.get('query', '')}：{json.dumps(obs.get('result', ''), ensure_ascii=False)[:300]}")
        else:
            parts.append(f"- {json.dumps(value, ensure_ascii=False)[:300]}")   # 兜底：直接截断
    return "\n".join(parts)


  react_writer_run 源码（行动 = llm_gen）
asyn

---
## Part 8：LangGraph 工作流 —— 把节点串成一张图
---

### 8.1 为什么需要 LangGraph？

我们已经有了：Planner（节点）、Scheduler（节点）、AgentState（状态）。
但还缺一个"编排器"来定义：

1. 节点之间怎么连接（谁先谁后、有没有分支/循环）
2. 状态怎么在节点间传递与合并
3. 图如何编译成可执行的执行器

**LangGraph** 就是官方编排框架，核心 API：

```python
from langgraph.graph import StateGraph, START, END

graph = StateGraph(AgentState)      # 1. 用状态类型建图
graph.add_node("planner", planner_node)   # 2. 加节点（函数：state -> 增量dict）
graph.add_edge(START, "planner")          # 3. 连边（控制流）
graph.add_edge("planner", "scheduler")
graph.add_edge("scheduler", END)
app = graph.compile()                     # 4. 编译成可执行图
```

> 生活类比：StateGraph 就像"流水线图纸"——图纸上画清楚每个工位做什么、
> 工件怎么流转；`compile()` 之后才是一条真正能跑的生产线。

> 💡 第 24 节是**线性版本**（无循环、无分支），第 25 节会加入 Reviewer 条件循环。

In [21]:
# ============================================================
# Cell 20：走读 graph/workflow.py —— build_graph 与 run_task
# ============================================================
import inspect

try:
    from graph import workflow
    print("=" * 55)
    print("  build_graph 源码")
    print("=" * 55)
    print(inspect.getsource(workflow.build_graph))
    print()
    print("=" * 55)
    print("  run_task 源码")
    print("=" * 55)
    print(inspect.getsource(workflow.run_task))
except Exception as e:
    print(f"  [提示] 导入失败（{e}）")

  build_graph 源码
def build_graph():
    """构建并编译 LangGraph（第 25 节：带 Reviewer/Rewriter 条件循环）。

    教学要点：
    - `add_conditional_edges("reviewer", 路由函数, 映射表)`：根据 state 动态决定下一步
    - 路由函数返回映射表里的 key（"passed"→END / "rewrite"→rewriter）
    - `add_edge("rewriter", "reviewer")`：返工后**循环回 Reviewer 再审核**（不重跑 Scheduler）
    """
    graph = StateGraph(AgentState)
    graph.add_node("planner", planner_node)          # 节点1：拆解计划
    graph.add_node("scheduler", scheduler_node)      # 节点2：按依赖派发执行
    graph.add_node("reviewer", reviewer_node)        # 节点3（25 节）：审核草稿
    graph.add_node("rewriter", rewriter_node)        # 节点4（25 节）：按意见返工
    graph.add_edge(START, "planner")
    graph.add_edge("planner", "scheduler")
    graph.add_edge("scheduler", "reviewer")          # 写完整稿 → 进入审核
    graph.add_conditional_edges(
        "reviewer",
        route_after_review,
        {"passed": END, "rewrite": "rewriter"},      # 通过 → 结束；不通过 → 返工
    )
    graph.add_edge("rewriter", "reviewer")           # 返工完 → 再次审核（条件

In [22]:
# ============================================================
# Cell 21：实际构建图并查看节点与边
# ============================================================
try:
    from graph import workflow

    app = workflow.build_graph()
    print("  [OK] LangGraph 编译成功！")
    print()
    print("  图中的节点：", list(app.get_graph().nodes.keys()))
    print("  图中的边：  ", list(app.get_graph().edges.keys()))
    print()
    print("  结构示意：")
    print("     START -> planner -> scheduler -> END")
    print()
    print("  [说明] 线性版本没有循环/分支，第 25 节会引入条件边")
except Exception as e:
    print(f"  [提示] 构建失败（{e}）")

  [OK] LangGraph 编译成功！

  图中的节点： ['__start__', 'planner', 'scheduler', 'reviewer', 'rewriter', '__end__']
  [提示] 构建失败（'list' object has no attribute 'keys'）


---
## Part 9：SQLite 持久化 —— 让任务"活下来"
---

### 9.1 痛点：重启即丢

如果把任务数据只放在内存里，后端一重启，历史任务、ReAct 日志全没了。
产品经理要求：**提交过的任务必须能随时查回来**。

解决方案：用 Python 内置的 `sqlite3`（零新增依赖），存到 `backend/data.db`。

### 9.2 两张表

```sql
tasks       -- 任务主表：task_id / topic / status / plan / completed_steps /
            --          research_data / draft_content / final_report / error / 时间戳
task_logs   -- ReAct 日志表：task_id / step_id / log_type(Thought|Action|Observation) / content
```

### 9.3 职责边界（重要）

| 库 | 存什么 | 谁写 |
|----|--------|------|
| `data.db`（本节） | **业务数据**：任务、ReAct 日志 | `task_repository` 显式写入 |
| `checkpoints.db`（第 26 节） | **图运行时状态**：LangGraph Checkpointer | 框架自动写入 |

两者互不干扰；`task_id` 同时作为 checkpoint 的 `thread_id`。

In [23]:
# ============================================================
# Cell 22：走读 db/database.py —— 连接管理与建表
# ============================================================
import inspect

try:
    from db import database
    print("=" * 55)
    print("  init_db 源码")
    print("=" * 55)
    print(inspect.getsource(database.init_db))
    print()
    print(f"  [说明] 业务库路径: {database.DB_PATH}")
except Exception as e:
    print(f"  [提示] 导入失败（{e}）")

  init_db 源码
def init_db(db_path: Path | None = None) -> None:
    """初始化数据库：可选覆盖库文件路径（测试用临时库），并执行建表 SQL（幂等）。

    :param db_path: 指定库文件路径；不传则用默认 backend/data.db
                    （测试/notebook 演示可通过它隔离出临时库，避免污染真实数据）
    """
    global DB_PATH
    if db_path is not None:
        DB_PATH = Path(db_path)             # 覆盖全局库路径（模块级单例）
    with sqlite3.connect(DB_PATH) as conn:
        conn.executescript(_SCHEMA)         # 批量执行建表 SQL（IF NOT EXISTS → 幂等）
        for sql in _MIGRATIONS:             # 旧库补列（已存在则跳过）
            try:
                conn.execute(sql)
            except sqlite3.OperationalError:
                pass


  [说明] 业务库路径: C:\Users\13691\Documents\Code\260601-GPKJ\05_multi-agent\proj04\smartbrief\backend\data.db


In [24]:
# ============================================================
# Cell 23：用"临时库"初始化并查看表结构（不碰真实 data.db）
# ============================================================
import os
import tempfile
import sqlite3

tmp_dir = tempfile.mkdtemp()
tmp_db = os.path.join(tmp_dir, "demo.db")
print(f"  临时库: {tmp_db}")
print()

from db import database
database.init_db(tmp_db)          # 用可选参数覆盖 DB_PATH（测试/演示用）
print("  [OK] 建表完成（幂等：重复执行也不会报错）")

conn = sqlite3.connect(tmp_db)
tables = conn.execute(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"
).fetchall()
print("  当前表：", [t[0] for t in tables])

print()
print("  tasks 表结构：")
for row in conn.execute("PRAGMA table_info(tasks)"):
    print(f"     {row[1]:20s} {row[2]}")

print()
print("  task_logs 表结构：")
for row in conn.execute("PRAGMA table_info(task_logs)"):
    print(f"     {row[1]:20s} {row[2]}")
conn.close()

  临时库: C:\Users\13691\AppData\Local\Temp\tmpfv45ypc2\demo.db

  [OK] 建表完成（幂等：重复执行也不会报错）
  当前表： ['sqlite_sequence', 'task_logs', 'tasks']

  tasks 表结构：
     task_id              TEXT
     topic                TEXT
     status               TEXT
     plan                 TEXT
     current_step_index   INTEGER
     completed_steps      TEXT
     research_data        TEXT
     draft_content        TEXT
     final_report         TEXT
     error                TEXT
     review_comments      TEXT
     review_history       TEXT
     iteration            INTEGER
     max_iterations       INTEGER
     passed               INTEGER
     forced_pass          INTEGER
     created_at           TEXT
     updated_at           TEXT

  task_logs 表结构：
     id                   INTEGER
     task_id              TEXT
     step_id              INTEGER
     log_type             TEXT
     content              TEXT
     created_at           TEXT


In [25]:
# ============================================================
# Cell 24：走读 db/task_repository.py —— 任务/日志 CRUD
# ============================================================
import inspect

try:
    from db import task_repository
    for fn_name in ["create_task", "update_task", "get_task", "append_log", "get_logs", "list_tasks"]:
        fn = getattr(task_repository, fn_name)
        print("=" * 55)
        print(f"  {fn_name} 源码")
        print("=" * 55)
        print(inspect.getsource(fn))
        print()
except Exception as e:
    print(f"  [提示] 导入失败（{e}）")

  create_task 源码
def create_task(task_id: str, topic: str) -> None:
    """插入一条任务初始记录（status 默认 running）。

    由 API 层在提交任务时调用（后台工作流启动之前），
    保证前端立刻能在任务列表里看到这条"执行中"记录。
    """
    with get_connection() as conn:
        conn.execute(
            "INSERT INTO tasks (task_id, topic) VALUES (?, ?)",
            (task_id, topic),
        )


  update_task 源码
def update_task(task_id: str, **fields) -> None:
    """更新任务字段（自动 JSON 序列化，并刷新 updated_at）。

    教学要点：**动态拼 SET 子句** —— 只更新传入的字段；
    结构化字段（plan 等）在写入前自动 _dumps 序列化。
    未在 _ALLOWED_UPDATE 白名单里的字段会被静默忽略，避免脏数据。

    :param task_id: 任务 id
    :param fields: 要更新的字段（keyword 参数）
    """
    updates = {k: v for k, v in fields.items() if k in _ALLOWED_UPDATE}   # 白名单过滤
    if not updates:
        return
    sets = []
    params: list = []
    for key, value in updates.items():
        sets.append(f"{key} = ?")
        # 结构化字段走 JSON 序列化，普通字段原样写入
        params.append(_dumps(value) if key in _JSON_FIELDS else value)
    sets.append("updated_at 

In [26]:
# ============================================================
# Cell 25：在临时库上演示 CRUD 全流程
# ============================================================
from db import task_repository

# 1. 创建任务（初始 status=running）
task_repository.create_task("task_crud_demo", "低空经济 2025 分析")
print("  [1] create_task 完成")

# 2. 更新任务（plan 等 JSON 字段自动序列化）
task_repository.update_task(
    "task_crud_demo",
    status="completed",
    plan=[{"id": 1, "agent": "researcher", "task": "搜索市场规模", "depends_on": []}],
    completed_steps=[1],
    draft_content="## 市场规模\n2025 年约 500 亿元",
)
print("  [2] update_task 完成（plan/completed_steps 已 JSON 序列化落库）")

# 3. 追加 ReAct 日志
task_repository.append_log("task_crud_demo", 1, "Thought", "开始研究")
task_repository.append_log("task_crud_demo", 1, "Action", "web_search(query='低空经济 市场规模')")
task_repository.append_log("task_crud_demo", 1, "Observation", "2025 年约 500 亿元")
print("  [3] append_log x3 完成")

# 4. 查询任务（JSON 字段自动反序列化）
task = task_repository.get_task("task_crud_demo")
print()
print("  [4] get_task 结果：")
print(f"     status        = {task['status']}")
print(f"     plan          = {task['plan']}")
print(f"     completed     = {task['completed_steps']}")
print(f"     draft_content = {task['draft_content'][:40]}...")

# 5. 查询日志
logs = task_repository.get_logs("task_crud_demo")
print()
print(f"  [5] get_logs：共 {len(logs)} 条")
for lg in logs:
    print(f"     [{lg['log_type']}] step{lg['step_id']}: {lg['content']}")

# 6. 列表查询（分页 + 过滤）
items, total = task_repository.list_tasks(status="completed", limit=10, offset=0)
print()
print(f"  [6] list_tasks(status='completed') -> total={total}, items={len(items)}")

  [1] create_task 完成
  [2] update_task 完成（plan/completed_steps 已 JSON 序列化落库）
  [3] append_log x3 完成

  [4] get_task 结果：
     status        = completed
     plan          = [{'id': 1, 'agent': 'researcher', 'task': '搜索市场规模', 'depends_on': []}]
     completed     = [1]
     draft_content = ## 市场规模
2025 年约 500 亿元...

  [5] get_logs：共 3 条
     [Thought] step1: 开始研究
     [Action] step1: web_search(query='低空经济 市场规模')
     [Observation] step1: 2025 年约 500 亿元

  [6] list_tasks(status='completed') -> total=1, items=1


---
## Part 10：API 与前端闭环
---

### 10.1 三个新接口

`api/routes.py` 新增三个路由，把后端能力暴露给前端：

| 方法 | 路径 | 功能 |
|------|------|------|
| POST | `/api/v1/generate` | 提交任务（创建记录 + 后台异步执行），立即返回 `task_id` |
| GET | `/api/v1/tasks` | 任务列表（分页 `page`/`page_size` + 状态过滤 `status`） |
| GET | `/api/v1/tasks/{task_id}/status` | 任务详情 + Plan 进度 + ReAct 日志（从 SQLite 读） |

**关键设计**：`/generate` 用 `asyncio.create_task()` 把 `workflow.run_task()` 扔到后台执行，
接口**立即返回** `task_id`；前端通过轮询 `/status` 实时刷新进度。

```python
@router.post("/generate")
async def generate_task(req: GenerateRequest):
    task_id = f"task_{uuid4().hex[:8]}"
    topic = req.topic.strip()
    task_repository.create_task(task_id, topic)          # 先落初始记录(running)
    asyncio.create_task(workflow.run_task(task_id, topic))  # 后台异步执行
    return {"task_id": task_id, "status": "running", "topic": topic}
```

### 10.2 前端三个页面

| 页面 | 作用 |
|------|------|
| `Home.vue` | 输入主题提交任务，成功后跳转详情；提供「历史任务」入口 |
| `Tasks.vue` | 任务列表：状态 Tab（全部/执行中/已完成/失败）+ 分页 + 「查看」按钮 |
| `TaskDetail.vue` | 左侧 Plan 看板 + 右侧 ReAct 日志流（2s 轮询）+ 报告草稿富文本 + 导出 Word/PDF |

> 前端通过 `fetchTaskStatus(taskId)` 每 2 秒轮询一次 `/tasks/{id}/status`，
> 只要任务还是 `running` 就继续刷新 —— 这就是"ReAct 日志逐条滚动"的实现原理。

In [27]:
# ============================================================
# Cell 26：走读 api/routes.py —— 任务管理部分
# ============================================================
import inspect

try:
    from api import routes
    print("=" * 55)
    print("  generate_task 源码")
    print("=" * 55)
    print(inspect.getsource(routes.generate_task))
    print()
    print("=" * 55)
    print("  list_tasks 源码")
    print("=" * 55)
    print(inspect.getsource(routes.list_tasks))
    print()
    print("=" * 55)
    print("  task_status 源码")
    print("=" * 55)
    print(inspect.getsource(routes.task_status))
except Exception as e:
    print(f"  [提示] 导入失败（{e}）")
    print("         这是 API 层，会连带加载技能池/MCP 等模块；")
    print("         不影响后续学习，接口逻辑已在上方贴出。")

  generate_task 源码
@router.post("/generate")
async def generate_task(req: GenerateRequest):
    """提交任务：创建任务记录并异步执行 LangGraph 工作流，立即返回 task_id。"""
    task_id = f"task_{uuid4().hex[:8]}"
    topic = req.topic.strip()
    # 先写入任务初始记录（running），再后台执行（前端轮询 status 可见）
    task_repository.create_task(task_id, topic)
    asyncio.create_task(workflow.run_task(task_id, topic))
    return {"task_id": task_id, "status": "running", "topic": topic}


  list_tasks 源码
@router.get("/tasks")
async def list_tasks(
    page: int = Query(1, ge=1, description="页码（从 1 开始）"),
    page_size: int = Query(10, ge=1, le=100, description="每页条数"),
    status: str | None = Query(default=None, description="按状态过滤：running / completed / failed"),
):
    """任务列表（分页 + 可选状态过滤），按创建时间倒序。"""
    items, total = task_repository.list_tasks(
        status=status, limit=page_size, offset=(page - 1) * page_size
    )
    return {"total": total, "page": page, "page_size": page_size, "items": items}


  task_status 源码
@router.get("/

---
## Part 11：端到端演示 —— 完整跑一遍（模拟 LLM，无需 API Key）
---

把前面所有零件拼起来：**真实 LangGraph 图 + 模拟的 LLM 与搜索工具 + 临时 SQLite 库**，
完整走一遍"关键词 → 计划 → ReAct 研究 → 撰写 → 落库"，观察每一条日志如何被持久化。

> 💡 这里用 `unittest.mock` 替换掉 `chat()` 和 `call_tool()`，
> 模拟 LLM 的"思考决策"，让我们在**不花钱、不联网**的情况下看到完整闭环。
> 真实运行时只需去掉 patch，换成真实的 `chat()` / MCP 工具即可。

In [28]:
# ============================================================
# Cell 27：端到端演示 —— 模拟 LLM 完整跑通 run_task
# ============================================================
import asyncio
import json
import os
import tempfile
from unittest.mock import patch, AsyncMock

# --- 准备临时库 ---
from db import database
from db import task_repository
database.init_db(os.path.join(tempfile.mkdtemp(), "e2e.db"))

# --- 模拟 LLM 的返回值 ---
class FakeMessage:
    def __init__(self, content):
        self.content = content

class FakeChoice:
    def __init__(self, content):
        self.message = FakeMessage(content)

class FakeResp:
    def __init__(self, content):
        self.choices = [FakeChoice(content)]

# Planner：返回一个 3 步计划
plan_text = json.dumps([
    {"id": 1, "agent": "researcher", "task": "搜索智能手表 2025 市场规模", "depends_on": []},
    {"id": 2, "agent": "researcher", "task": "搜索智能手表头部厂商",       "depends_on": []},
    {"id": 3, "agent": "writer",     "task": "撰写市场调研报告草稿",       "depends_on": [1, 2]},
], ensure_ascii=False)

# Researcher：第 1 步搜市场规模->done，第 2 步搜头部厂商->done
researcher_decisions = [
    FakeResp(json.dumps({"action": "web_search", "query": "智能手表 2025 市场规模"})),
    FakeResp(json.dumps({"action": "done"})),
    FakeResp(json.dumps({"action": "web_search", "query": "智能手表 头部厂商"})),
    FakeResp(json.dumps({"action": "done"})),
]


In [29]:

# 搜索工具：返回预设数据
def fake_call_tool_side(name, args):
    query = args.get("query", "")
    if "头部厂商" in query:
        return {"content": "Apple、华为、三星、小米占据前四"}
    return {"content": "2025 年全球智能手表市场规模约 300 亿美元，年增 15%"}

fake_planner_chat = AsyncMock(return_value=FakeResp(plan_text))
fake_researcher_chat = AsyncMock(side_effect=researcher_decisions)
fake_writer_chat = AsyncMock(return_value=FakeResp("## 市场规模\n\n2025 年全球智能手表市场规模约 300 亿美元。\n\n## 竞争格局\n\nApple、华为、三星、小米占据前四。"))
fake_call_tool = AsyncMock(side_effect=fake_call_tool_side)

# --- 提交任务（API 层职责：先建初始记录） ---
task_id = "task_e2e_01"
task_repository.create_task(task_id, "智能手表 2025 市场")
print(f"  [1] 任务已创建: {task_id} (status=running)")
print()

# --- 跑真实图（patch 掉 LLM 与搜索） ---
from graph import workflow
from agents import planner, react_researcher, react_writer

print("  [2] 开始执行 LangGraph 工作流（模拟 LLM）...")
print()

with (
    patch.object(planner, "chat", fake_planner_chat),
    patch.object(react_researcher, "chat", fake_researcher_chat),
    patch.object(react_writer, "chat", fake_writer_chat),
    patch.object(react_researcher.mcp_manager, "call_tool", fake_call_tool),
):
    # Jupyter 内核已运行在事件循环中：不能用 asyncio.run，直接 await
    # （若在普通 .py 脚本中运行，可改回 asyncio.run(workflow.run_task(...))）
    await workflow.run_task(task_id, "智能手表 2025 市场")

print("  [3] 工作流执行完毕，正在落库...")
print()

# --- 验证落库结果 ---
task = task_repository.get_task(task_id)
print(f"  [4] 任务状态: {task['status']}")
print(f"      Plan 步骤: {len(task['plan'])} 个")
print(f"      已完成步骤: {task['completed_steps']}")
print(f"      草稿内容: {task['draft_content'][:60]}...")
print()

logs = task_repository.get_logs(task_id)
print(f"  [5] ReAct 日志已持久化，共 {len(logs)} 条：")
print("  " + "-" * 60)
for lg in logs:
    print(f"     [{lg['log_type']:11s}] step{lg['step_id']}: {lg['content'][:48]}")
print("  " + "-" * 60)
print()
print("  [结论] 这就是后端真实发生的一切：")
print("         任务 -> Planner 拆解 -> Scheduler 派发 -> ReAct 研究 -> 撰写")
print("         每一条 Thought/Action/Observation 都进了 task_logs 表")

  [1] 任务已创建: task_e2e_01 (status=running)

  [2] 开始执行 LangGraph 工作流（模拟 LLM）...

  [3] 工作流执行完毕，正在落库...

  [4] 任务状态: completed
      Plan 步骤: 3 个
      已完成步骤: [1, 2, 3]
      草稿内容: ## 市场规模

2025 年全球智能手表市场规模约 300 亿美元。

## 竞争格局

Apple、华为、三星、小米...

  [5] ReAct 日志已持久化，共 23 条：
  ------------------------------------------------------------
     [Thought    ] step0: 已生成执行计划：3 个步骤（researcher, researcher, writer）
     [Thought    ] step1: 调度：步骤 1 派发给 researcher
     [Thought    ] step1: 第 1 轮：{"action": "web_search", "query": "\u667a\
     [Action     ] step1: web_search(query='智能手表 2025 市场规模')
     [Observation] step1: {"content": "2025 年全球智能手表市场规模约 300 亿美元，年增 15%"}
     [Thought    ] step1: 第 2 轮：{"action": "done"}
     [Thought    ] step1: 信息已足够，结束本轮研究
     [Observation] step1: 步骤 1 完成
     [Thought    ] step2: 调度：步骤 2 派发给 researcher
     [Thought    ] step2: 第 1 轮：{"action": "web_search", "query": "\u667a\
     [Action     ] step2: web_search(query='智能手表 头部厂商')
     [Observation] step2: {"con

---
## 课程总结
---

### 我们经历了什么？

| 演进阶段 | 遇到的问题 | 解决方案 |
|---------|-----------|---------|
| v0 | 一个大 Prompt 生成报告，质量差、会编数据 | 拆成多个 Agent 分工协作 |
| v1 | 多个 Agent 之间怎么传数据 | 共享状态 `AgentState`（白板） |
| 规划线 | 硬编码计划死板，无法适配任意主题 | LLM 生成 + `parse_plan` 容错解析 + `fallback_plan` 降级 |
| 调度线 | 顺序执行会让 Writer 先于 Researcher 跑 | `depends_on` 依赖就绪判定 + 逐轮派发 |
| 执行线 | 搜索一次不够、不知道何时停 | ReAct 循环：Thought→Action→Observation（最多 3 轮） |
| 编排 | 节点/状态/控制流需要统一管理 | LangGraph `StateGraph` 建图 + `compile()` |
| 持久化 | 重启即丢，任务无法回溯 | SQLite `tasks` / `task_logs` 两张表 |

### 关键设计原则

1. **单一职责**：Planner / Scheduler / Researcher / Writer 各管一件事
2. **共享状态**：多智能体通过 `AgentState` 沟通，节点只返回增量
3. **容错优先**：LLM 不可靠 → 容错解析 + 降级计划，流程永远可跑
4. **过程可见**：每一步 ReAct 都记日志并落库，前端实时轮询展示
5. **先跑通再优化**：第 24 节是线性版本，条件循环（Reviewer）、并行执行留待后续

### 下节课预告

**第 25 节：A2A 协同与反馈循环**
- 基于共享 State + 条件边实现"Reviewer 审查 → 打回重写"的反馈闭环
- 条件路由（`add_conditional_edges`）、循环控制
- 多智能体协作的"质量保障"机制

---

## 课后练习

### 练习一：你是架构师 —— 加入 Reviewer

产品经理说："报告写完要有人**审核**，不合格就**打回重写**，最多重写 2 次。"

请思考：
1. 需要在 `AgentState` 里加哪些字段（如 `review_count`、`needs_revision`）？
2. LangGraph 的哪条边需要从"固定边"变成"条件边"？
3. 第 25 节会给出的实现方案是什么？可以先去 `graph/workflow.py` 里试试。

> 提示：条件边的函数返回 `"accept"` 或 `"reject"`，LangGraph 据此选择下一个节点。

### 练习二：给 Researcher 加一个"第 4 轮"

修改 `MAX_REACT_STEPS` 的语义：假设搜索 3 次还不够，需要允许最多 5 轮。
- 修改哪个常量？（`agents/react_researcher.py`）
- 无限放宽会带来什么问题？（成本、上下文长度、死循环）

### 练习三：观察真实运行

配置好 `backend/.env`（API Key）后：
1. 启动后端：`uvicorn main:app --reload --port 8002`
2. 打开 http://localhost:5174/home 提交一个主题
3. 在任务详情页观察：Plan 看板逐步点亮、ReAct 日志逐条滚动
4. 完成后导出 Word / PDF，对比草稿的 Markdown 源文与富文本渲染
5. 重启后端，到「任务列表」页确认历史任务仍在（SQLite 持久化生效）

---

## 参考资料

- 规范：`多智能体协作开发PRD（拆分）.md` —— 子文档 3（第 24 节）
- 代码：`backend/graph/state.py` —— AgentState
- 代码：`backend/graph/workflow.py` —— LangGraph 工作流
- 代码：`backend/agents/planner.py` / `scheduler.py` / `react_researcher.py` / `react_writer.py`
- 代码：`backend/db/database.py` / `task_repository.py` —— SQLite 持久化
- 官方：LangGraph 文档 https://langchain-ai.github.io/langgraph/

> 🎉 恭喜你完成了本节课的学习！
> 现在你不仅知道多智能体代码长什么样，更知道它**为什么长成这样**。